# measured vs allowed separation of NEA ID matches
- ID-matched planets sorted by crossmatching distance; minimum (10'') and fallback (50'') radii
- exports to `report/figures/nea_coordinate_calibration.pdf`

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'crossmatching').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u

from crossmatching import NEACatalog, allowed_angular_separation
from report_plots import common, data


In [ ]:
cm, input_t, matched = data.nea_crossmatch()
id_matched = matched[matched["match_type"] != 'coordinates']
id_matched = cm.remove_duplicates(id_matched, "star_name", from_file=str(data.ALTERNATE_IDS_PATH))
id_matched.sort("crossmatching_angular_separation")
len(id_matched)


In [ ]:
id_matched["expected_sep"] = allowed_angular_separation(
    proper_motion=id_matched["sy_pm"]*NEACatalog().pm_unit.to(u.arcsec/u.yr),
    pm_err=id_matched["sy_pmerr1"]*NEACatalog().pm_unit.to(u.arcsec/u.yr),
    epoch=id_matched["coord_epoch"],
    input_epoch=2000,
    minimum = 0 * u.arcsec,
    unknown_default=0 * u.arcsec,
)

id_matched["allowed_sep"] = allowed_angular_separation(
    proper_motion=id_matched["sy_pm"]*NEACatalog().pm_unit.to(u.arcsec/u.yr),
    pm_err=id_matched["sy_pmerr1"]*NEACatalog().pm_unit.to(u.arcsec/u.yr),
    epoch=id_matched["coord_epoch"],
    input_epoch=2000,
    unknown_default=np.nan * u.arcsec
)

In [ ]:
x = list(range(len(id_matched)))
id_matched.sort("crossmatching_angular_separation")

# fig, ax = plt.subplots(figsize=(7, 4))
fig, ax = plt.subplots(figsize=(5, 3))
ax.axhline(50, color="dimgray", ls="--", lw=1)
ax.axhline(10, color="gray", ls="--", lw=1)
ax.set_yscale("log")

from matplotlib.ticker import FixedFormatter, FixedLocator

ax.set_ylim(0.01, 1000)
tick_values = [1E-2, 1E-1, 1, 10, 50, 100, 1000]
ax.yaxis.minorticks_off()
ax.yaxis.set_major_locator(FixedLocator(tick_values))
ax.yaxis.set_major_formatter(FixedFormatter([f"{i}" for i in tick_values]))
ax.xaxis.set_major_formatter(FixedFormatter([]))
plt.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)

measured_vals = np.asarray(id_matched["crossmatching_angular_separation"])


index_below_default = np.argmin(id_matched["crossmatching_angular_separation"] < 50 * u.arcsec)
index_below_minimum = np.argmin(id_matched["crossmatching_angular_separation"] < 10 * u.arcsec)

x_fill_default = x[:index_below_default]
y_fill_default = measured_vals[:index_below_default]
x_fill_min = x[:index_below_minimum]
y_fill_min = measured_vals[:index_below_minimum]

ax.fill_between(x_fill_default, y_fill_default, y2=0, step="mid", color="#aaaaaa")
ax.text(
    (x_fill_min[-1] + x_fill_default[-1]) /2,
    0.05,
    f"Below\nfallback\n{(index_below_default+1)/len(id_matched)*100:.1f}%",
    ha="center",
    va="bottom",
    fontsize=10,
)

ax.fill_between(x_fill_min, y_fill_min, y2=0, step="mid", color="lightgray")
ax.text(
    (x_fill_min[0] + x_fill_min[-1]) / 2,
    0.05,
    f"Below\nminimum\n{(index_below_minimum+1)/len(id_matched)*100:.1f}%",
    ha="center",
    va="bottom",
    fontsize=10,
)

ax.step(x, id_matched["crossmatching_angular_separation"], where="pre", label="Measured separation", lw=2)
ax.step(x, id_matched["allowed_sep"], where="pre", label="Allowed separation", lw=0.75)

ax.text(len(id_matched)*0.1, 10 -1 , "minimum: 10''", ha="left", va="top", color="gray")
ax.text(len(id_matched)*0.1, 50 +1 , "fallback: 50''", ha="left", va="bottom", color="dimgray")

ax.set_xlim(-5,len(id_matched) + 2)
ax.set_ylabel("Crossmatching Distance [arcsec]")
ax.set_xlabel("ordered by inceasing crossmatching distance")
ax.legend(loc="upper right", bbox_to_anchor=(0.9,1))

common.save_figure("nea_coordinate_calibration")